# HMM-based Ventilation Phase Inference & 1-Hour Forecasting

This notebook infers a patient's ventilation **phase** (4-state Hidden Markov Model) each hour after intubation and forecasts the phase 1 hour ahead.

## Clinical Phases (Hidden States)
| State | Name | Clinical Interpretation |
|-------|------|-------------------------|
| S0 | **Acute** | Assist/Control–like, higher support/instability |
| S1 | **Recovery** | PSV/PAV+–like, improving but supported |
| S2 | **Weaning** | pre-SBT/SBT–like, low support + stable |
| S3 | **Liberation** | Extubation/vent-off proxy |

## ⚠️ Important Data Limitations
We do **NOT** have: FiO2, SpO2, PaO2, pH, vasopressor dose, ventilator mode, PS, or SBT results.

We **DO** have: PEEP, Peak pressure, MAP/SBP/DBP, labs (WBC, CRP, creatinine, etc.), temperature.

**Therefore, the HMM phases are "latent course stages" correlated with these available proxies.**

## 0. Setup & Configuration

In [ ]:
# Dependencies are in pixi.toml - run with:
# pixi run jupyter lab
# Or run the script directly: pixi run python Agent_B/hmm_ventilation_phase_inference.py

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✅ All imports successful!")

In [ ]:
# =============================================================================
# CONFIGURATION - Easy to tweak parameters
# =============================================================================

# File paths (use parquet for faster loading)
INPUT_FILE = "../clinical_data/data_v1_max_72_h.parquet"
OUTPUT_CSV = "hmm_phase_predictions.csv"

# Phase label names
PHASE_NAMES = {0: "Acute", 1: "Recovery", 2: "Weaning", 3: "Liberation"}
PHASE_COLORS = {'Acute': '#d62728', 'Recovery': '#ff7f0e', 'Weaning': '#2ca02c', 'Liberation': '#1f77b4'}

# Weak label heuristic cutoffs
PEEP_ACUTE_THRESHOLD = 12       # PEEP >= this suggests Acute
PEEP_WEANING_THRESHOLD = 8      # PEEP <= this suggests Weaning-ready
PEAK_ACUTE_THRESHOLD = 30       # Peak pressure >= this suggests Acute
MAP_UNSTABLE_THRESHOLD = 65     # MAP < this suggests hemodynamic instability

# Missing data handling
MAX_FORWARD_FILL_HOURS = 4

# HMM configuration
N_COMPONENTS = 4
COVARIANCE_TYPE = "full"
N_ITER = 100
RANDOM_STATE = 42

# Transition matrix: encourages forward progression (Acute -> Recovery -> Weaning -> Liberation)
INIT_TRANSMAT = np.array([
    # To:  Acute, Recovery, Weaning, Liberation
    [0.85,  0.12,    0.02,     0.01],  # From Acute
    [0.08,  0.82,    0.08,     0.02],  # From Recovery (can deteriorate back)
    [0.02,  0.08,    0.82,     0.08],  # From Weaning
    [0.01,  0.02,    0.07,     0.90],  # From Liberation (mostly absorbing)
])

# Feature configuration
BASE_FEATURES = [
    'peep_mean', 'peak_mean', 'map_mean', 'sbp_mean', 'dbp_mean', 'temp_mean',
    'wbc_mean', 'crp_mean', 'creatinine_mean', 'glucose_mean',
    'sodium_mean', 'potassium_mean', 'chloride_mean',
    'hemoglobin_mean', 'platelets_mean'
]

TREND_FEATURES = ['peep_mean', 'peak_mean', 'map_mean']
MISSINGNESS_FEATURES = ['peep_mean', 'peak_mean']

print("Configuration loaded!")
print(f"  Input file: {INPUT_FILE}")
print(f"  HMM components: {N_COMPONENTS}")
print(f"  Base features: {len(BASE_FEATURES)}")

## 1. Load & Prepare Data

In [ ]:
def detect_time_format(measure_time: pd.Series) -> str:
    """
    Detect time format:
    - epoch_seconds: Unix timestamp (~10 digits, > 1e9)
    - epoch_ms: Unix timestamp in ms (~13 digits, > 1e12)
    - hour_index: Small integers representing hours
    """
    sample_vals = measure_time.dropna().head(100)
    if len(sample_vals) == 0:
        return 'hour_index'
    
    median_val = sample_vals.median()
    
    if median_val > 1e12:
        return 'epoch_ms'
    elif median_val > 1e9:
        return 'epoch_seconds'
    else:
        return 'hour_index'


def load_and_prepare(filepath: str) -> pd.DataFrame:
    """
    Load data and create hourly time index per visit.
    
    For each visit_occurrence_id:
    - Sort by measure_time
    - Convert time appropriately
    - Create t_hr (hours since first measurement)
    - Resample to 1-hour bins if needed
    """
    print(f"Loading data from {filepath}...")
    
    if filepath.endswith('.parquet'):
        df = pd.read_parquet(filepath)
    else:
        df = pd.read_csv(filepath)
    
    # Handle multi-index (visit_occurrence_id, measure_time) from parquet
    if isinstance(df.index, pd.MultiIndex):
        print("  Detected multi-index, resetting...")
        df = df.reset_index()
    
    # Ensure required columns exist
    if 'visit_occurrence_id' not in df.columns:
        raise ValueError("Data must have 'visit_occurrence_id' column")
    if 'measure_time' not in df.columns:
        raise ValueError("Data must have 'measure_time' column")
    
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")
    print(f"  Unique visits: {df['visit_occurrence_id'].nunique():,}")
    
    # Detect time format
    time_format = detect_time_format(df['measure_time'])
    print(f"  Time format detected: {time_format}")
    
    processed_visits = []
    
    for visit_id, visit_df in df.groupby('visit_occurrence_id'):
        visit_df = visit_df.sort_values('measure_time').copy()
        
        if time_format == 'epoch_seconds':
            visit_df['datetime'] = pd.to_datetime(visit_df['measure_time'], unit='s', errors='coerce')
            first_time = visit_df['datetime'].min()
            visit_df['t_hr'] = (visit_df['datetime'] - first_time).dt.total_seconds() / 3600.0
        elif time_format == 'epoch_ms':
            visit_df['datetime'] = pd.to_datetime(visit_df['measure_time'], unit='ms', errors='coerce')
            first_time = visit_df['datetime'].min()
            visit_df['t_hr'] = (visit_df['datetime'] - first_time).dt.total_seconds() / 3600.0
        else:  # hour_index
            first_time = visit_df['measure_time'].min()
            visit_df['t_hr'] = visit_df['measure_time'] - first_time
        
        # Round to hourly bins
        visit_df['t_hr_bin'] = visit_df['t_hr'].round().astype(int)
        
        # Resample if multiple rows per hour
        if visit_df.groupby('t_hr_bin').size().max() > 1:
            numeric_cols = visit_df.select_dtypes(include=[np.number]).columns.tolist()
            agg_cols = [c for c in numeric_cols if c not in ['t_hr_bin', 'measure_time', 't_hr']]
            
            first_row = visit_df.groupby('t_hr_bin').first()[['visit_occurrence_id', 'person_id']].reset_index()
            agg_df = visit_df.groupby('t_hr_bin')[agg_cols].median().reset_index()
            visit_df = first_row.merge(agg_df, on='t_hr_bin')
            visit_df['t_hr'] = visit_df['t_hr_bin'].astype(float)
        
        processed_visits.append(visit_df)
    
    df_prepared = pd.concat(processed_visits, ignore_index=True)
    print(f"  After hourly resampling: {len(df_prepared):,} rows")
    
    return df_prepared

In [ ]:
# Load the data
df_raw = load_and_prepare(INPUT_FILE)

# Quick preview
print("\nData preview:")
display(df_raw.head())

# Check available columns
print("\nAvailable columns:")
print([c for c in df_raw.columns if any(x in c.lower() for x in ['peep', 'peak', 'map', 'wbc', 'crp', 'temp'])])

## 2. Feature Engineering

In [ ]:
def make_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Engineer features for HMM:
    - Base features (vitals, pressures, labs)
    - Trend features: 1-hour delta and 4-hour rolling slope
    - Missingness indicators (important for Liberation detection)
    """
    print("Engineering features...")
    
    df = df.copy()
    feature_cols = []
    
    # 1. Base features - use only available ones
    available_base = [f for f in BASE_FEATURES if f in df.columns]
    feature_cols.extend(available_base)
    print(f"  Base features available: {len(available_base)}/{len(BASE_FEATURES)}")
    
    # 2. Trend features per visit
    for feat in TREND_FEATURES:
        if feat not in df.columns:
            continue
        
        # 1-hour delta: x[t] - x[t-1]
        delta_col = f"{feat}_delta1h"
        df[delta_col] = df.groupby('visit_occurrence_id')[feat].diff(1)
        feature_cols.append(delta_col)
        
        # 4-hour slope: (x[t] - x[t-4]) / 4
        slope_col = f"{feat}_slope4h"
        df[slope_col] = df.groupby('visit_occurrence_id')[feat].transform(
            lambda x: x.diff(4) / 4
        )
        feature_cols.append(slope_col)
    
    trend_count = len([f for f in feature_cols if 'delta' in f or 'slope' in f])
    print(f"  Trend features added: {trend_count}")
    
    # 3. Missingness indicators
    for feat in MISSINGNESS_FEATURES:
        if feat not in df.columns:
            continue
        miss_col = f"{feat}_missing"
        df[miss_col] = df[feat].isna().astype(float)
        feature_cols.append(miss_col)
    
    miss_count = len([f for f in feature_cols if 'missing' in f])
    print(f"  Missingness indicators: {miss_count}")
    print(f"  Total features: {len(feature_cols)}")
    
    return df, feature_cols

In [ ]:
# Apply feature engineering
df, feature_cols = make_features(df_raw)

# Filter to available features
feature_cols = [f for f in feature_cols if f in df.columns]
print(f"\nFinal feature set ({len(feature_cols)} features):")
print(feature_cols)

## 3. Missing Data Handling

In [ ]:
def handle_missing_data(
    df: pd.DataFrame, 
    feature_cols: List[str],
    train_mask: Optional[pd.Series] = None
) -> Tuple[pd.DataFrame, StandardScaler, Dict[str, float]]:
    """
    Handle missing data:
    1. Forward-fill within each visit (up to MAX_FORWARD_FILL_HOURS)
    2. Impute remaining with global median (fit on train only)
    3. Standardize features
    """
    print("Handling missing data...")
    
    df = df.copy()
    
    # Count missing before
    missing_before = df[feature_cols].isna().sum().sum()
    print(f"  Missing values before: {missing_before:,}")
    
    # 1. Forward-fill within each visit
    for col in feature_cols:
        if col in df.columns:
            df[col] = df.groupby('visit_occurrence_id')[col].transform(
                lambda x: x.ffill(limit=MAX_FORWARD_FILL_HOURS)
            )
    
    missing_after_ffill = df[feature_cols].isna().sum().sum()
    print(f"  After forward-fill: {missing_after_ffill:,}")
    
    # 2. Compute global medians from training data
    if train_mask is None:
        train_mask = pd.Series(True, index=df.index)
    
    medians = {}
    for col in feature_cols:
        if col in df.columns:
            medians[col] = df.loc[train_mask, col].median()
            if pd.isna(medians[col]):
                medians[col] = 0.0
    
    # 3. Impute remaining missing
    for col in feature_cols:
        if col in df.columns:
            df[col] = df[col].fillna(medians[col])
    
    missing_after = df[feature_cols].isna().sum().sum()
    print(f"  After median imputation: {missing_after:,}")
    
    # 4. Standardize
    scaler = StandardScaler()
    scaler.fit(df.loc[train_mask, feature_cols])
    df[feature_cols] = scaler.transform(df[feature_cols])
    
    print(f"  Features standardized ✓")
    
    return df, scaler, medians

In [ ]:
# Train/test split by visit (80/20)
visits = df['visit_occurrence_id'].unique()
np.random.seed(RANDOM_STATE)
np.random.shuffle(visits)
split_idx = int(0.8 * len(visits))
train_visits = set(visits[:split_idx])
test_visits = set(visits[split_idx:])

train_mask = df['visit_occurrence_id'].isin(train_visits)
print(f"Train visits: {len(train_visits):,}")
print(f"Test visits: {len(test_visits):,}")
print(f"Train rows: {train_mask.sum():,}")
print(f"Test rows: {(~train_mask).sum():,}")

In [ ]:
# Handle missing data
df, scaler, medians = handle_missing_data(df, feature_cols, train_mask)

## 4. Weak Label Initialization

In [ ]:
def create_weak_labels(df: pd.DataFrame) -> pd.Series:
    """
    Create heuristic labels to seed HMM initialization.
    
    NOTE: These labels are rough proxies based on available data.
    The HMM will refine these based on observation patterns.
    
    Labels:
    - 0 (Acute): high PEEP OR high peak OR low MAP
    - 1 (Recovery): default intermediate state
    - 2 (Weaning): low PEEP AND stable MAP AND moderate peak
    - 3 (Liberation): missing vent measurements (proxy for extubation)
    """
    print("Creating weak labels for HMM initialization...")
    
    # Use raw (pre-standardized) data for heuristics if available
    # We'll work with standardized data but adjust thresholds
    
    labels = pd.Series(1, index=df.index)  # Default: Recovery
    
    # Get peak cutoff (75th percentile)
    if 'peep_mean' in df.columns and 'peak_mean' in df.columns:
        # For standardized data, use z-score thresholds
        # After standardization, median ≈ 0, so we use relative thresholds
        peep_acute_z = 0.5   # ~upper 30%
        peep_weaning_z = -0.3  # ~lower 40%
        peak_acute_z = 0.5
        map_unstable_z = -0.5  # lower than median
    else:
        peep_acute_z = 0.5
        peep_weaning_z = -0.3
        peak_acute_z = 0.5
        map_unstable_z = -0.5
    
    # Check for missingness indicators (Liberation proxy)
    if 'peep_mean_missing' in df.columns:
        lib_mask = df['peep_mean_missing'] > 0.5
        labels[lib_mask] = 3
        print(f"  Liberation-like (missing vent): {lib_mask.sum():,} rows")
    
    # Acute-like: high support or instability
    acute_mask = pd.Series(False, index=df.index)
    if 'peep_mean' in df.columns:
        acute_mask = acute_mask | (df['peep_mean'] > peep_acute_z)
    if 'peak_mean' in df.columns:
        acute_mask = acute_mask | (df['peak_mean'] > peak_acute_z)
    if 'map_mean' in df.columns:
        acute_mask = acute_mask | (df['map_mean'] < map_unstable_z)
    
    acute_mask = acute_mask & (labels != 3)  # Don't override Liberation
    labels[acute_mask] = 0
    print(f"  Acute-like: {acute_mask.sum():,} rows")
    
    # Weaning-like: low support + stable
    weaning_mask = pd.Series(True, index=df.index)
    if 'peep_mean' in df.columns:
        weaning_mask = weaning_mask & (df['peep_mean'] < peep_weaning_z)
    if 'map_mean' in df.columns:
        weaning_mask = weaning_mask & (df['map_mean'] > map_unstable_z)
    if 'peak_mean' in df.columns:
        weaning_mask = weaning_mask & (df['peak_mean'] < 0)  # Below median
    
    weaning_mask = weaning_mask & (labels == 1)  # Only from Recovery
    labels[weaning_mask] = 2
    print(f"  Weaning-like: {weaning_mask.sum():,} rows")
    
    recovery_count = (labels == 1).sum()
    print(f"  Recovery-like: {recovery_count:,} rows")
    
    return labels

In [ ]:
# Create weak labels (only on training data for initialization)
weak_labels = create_weak_labels(df[train_mask])

# Visualize distribution
fig, ax = plt.subplots(figsize=(8, 4))
label_counts = weak_labels.value_counts().sort_index()
bars = ax.bar([PHASE_NAMES[i] for i in label_counts.index], label_counts.values,
              color=[PHASE_COLORS[PHASE_NAMES[i]] for i in label_counts.index])
ax.set_ylabel('Count')
ax.set_title('Weak Label Distribution (Training Data)')
for bar, count in zip(bars, label_counts.values):
    ax.annotate(f'{count:,}', xy=(bar.get_x() + bar.get_width()/2, count),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 5. HMM Model Fitting

In [ ]:
def init_hmm_from_weak_labels(
    X_train: np.ndarray,
    weak_labels: np.ndarray,
    n_components: int = 4
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Initialize HMM means and covariances from weak labels.
    Falls back to KMeans if weak labels don't cover all components.
    """
    print("Initializing HMM parameters...")
    
    n_features = X_train.shape[1]
    means = np.zeros((n_components, n_features))
    covars = np.zeros((n_components, n_features, n_features))
    
    label_counts = pd.Series(weak_labels).value_counts()
    print(f"  Weak label counts: {dict(label_counts)}")
    
    use_kmeans = False
    for i in range(n_components):
        mask = weak_labels == i
        if mask.sum() < 10:
            print(f"  ⚠️ Label {i} ({PHASE_NAMES[i]}) has only {mask.sum()} samples")
            use_kmeans = True
            break
        
        means[i] = X_train[mask].mean(axis=0)
        covars[i] = np.cov(X_train[mask].T) + 1e-3 * np.eye(n_features)
    
    if use_kmeans:
        print("  Using KMeans fallback...")
        kmeans = KMeans(n_clusters=n_components, random_state=RANDOM_STATE, n_init=10)
        kmeans.fit(X_train)
        
        for i in range(n_components):
            mask = kmeans.labels_ == i
            means[i] = X_train[mask].mean(axis=0)
            covars[i] = np.cov(X_train[mask].T) + 1e-3 * np.eye(n_features)
    
    print("  ✓ Initialization complete")
    return means, covars

In [ ]:
def fit_hmm(
    X_sequences: List[np.ndarray],
    lengths: List[int],
    means_init: Optional[np.ndarray] = None,
    covars_init: Optional[np.ndarray] = None,
    fix_transmat: bool = False
) -> GaussianHMM:
    """
    Fit Gaussian HMM on concatenated sequences.
    
    The transition matrix is initialized to encourage forward progression
    through clinical phases (Acute -> Recovery -> Weaning -> Liberation).
    """
    print("Fitting HMM...")
    
    X_concat = np.vstack(X_sequences)
    print(f"  Training samples: {len(X_concat):,}")
    print(f"  Number of sequences: {len(lengths):,}")
    print(f"  Average sequence length: {np.mean(lengths):.1f} hours")
    
    # Initialize model
    model = GaussianHMM(
        n_components=N_COMPONENTS,
        covariance_type=COVARIANCE_TYPE,
        n_iter=N_ITER,
        random_state=RANDOM_STATE,
        verbose=False
    )
    
    # Set initial parameters
    model.startprob_ = np.array([0.7, 0.2, 0.08, 0.02])  # Most start in Acute
    model.transmat_ = INIT_TRANSMAT.copy()
    
    if means_init is not None:
        model.means_ = means_init
    if covars_init is not None:
        model.covars_ = covars_init
    
    # Control parameter updates
    if fix_transmat:
        model.params = 'mc'  # Only update means and covariances
        model.init_params = ''
        print("  Transition matrix: FIXED")
    else:
        model.params = 'stmc'
        model.init_params = ''
        print("  Transition matrix: LEARNABLE")
    
    # Fit
    model.fit(X_concat, lengths)
    
    print(f"\n  Converged: {model.monitor_.converged}")
    print(f"  Final log-likelihood: {model.score(X_concat, lengths):,.2f}")
    
    return model

In [ ]:
# Prepare training sequences
train_sequences = []
train_lengths = []
train_weak_labels_list = []

for visit_id in train_visits:
    visit_df = df[df['visit_occurrence_id'] == visit_id].sort_values('t_hr')
    if len(visit_df) < 2:  # Skip very short visits
        continue
    X = visit_df[feature_cols].values
    train_sequences.append(X)
    train_lengths.append(len(X))
    # Get weak labels for this visit
    visit_weak = weak_labels.loc[visit_df.index].values
    train_weak_labels_list.extend(visit_weak)

print(f"Prepared {len(train_sequences):,} training sequences")
print(f"Sequence length stats: min={min(train_lengths)}, max={max(train_lengths)}, median={np.median(train_lengths):.0f}")

In [ ]:
# Initialize from weak labels
X_train_concat = np.vstack(train_sequences)
train_weak_labels_arr = np.array(train_weak_labels_list)

means_init, covars_init = init_hmm_from_weak_labels(
    X_train_concat, train_weak_labels_arr, N_COMPONENTS
)

In [ ]:
# Fit HMM
# Set fix_transmat=True to keep clinically-motivated transitions, or False to learn from data
model = fit_hmm(
    train_sequences,
    train_lengths,
    means_init=means_init,
    covars_init=covars_init,
    fix_transmat=False  # Allow learning but start from clinical prior
)

In [ ]:
# Visualize learned transition matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(model.transmat_, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N_COMPONENTS))
ax.set_yticks(range(N_COMPONENTS))
ax.set_xticklabels([PHASE_NAMES[i] for i in range(N_COMPONENTS)])
ax.set_yticklabels([PHASE_NAMES[i] for i in range(N_COMPONENTS)])
ax.set_xlabel('To State')
ax.set_ylabel('From State')
ax.set_title('Learned Transition Probabilities')

# Add text annotations
for i in range(N_COMPONENTS):
    for j in range(N_COMPONENTS):
        text = ax.text(j, i, f'{model.transmat_[i, j]:.2f}',
                       ha='center', va='center', color='black' if model.transmat_[i, j] < 0.5 else 'white')

plt.colorbar(im, ax=ax, label='Probability')
plt.tight_layout()
plt.show()

# Print as table
print("\nTransition Matrix:")
print(pd.DataFrame(model.transmat_, 
                   index=[PHASE_NAMES[i] for i in range(N_COMPONENTS)],
                   columns=[PHASE_NAMES[i] for i in range(N_COMPONENTS)]).round(3))

## 6. Inference & 1-Hour Forecasting

In [ ]:
def infer_and_forecast(
    model: GaussianHMM,
    df: pd.DataFrame,
    feature_cols: List[str]
) -> pd.DataFrame:
    """
    For each visit:
    - Compute filtered posterior P(S_t | x_0:t) via forward algorithm
    - Compute 1-hour-ahead forecast: P(S_{t+1} | x_0:t) = alpha_t @ transmat
    
    Returns DataFrame with predictions for each timepoint.
    """
    print("Running inference and forecasting...")
    
    results = []
    transmat = model.transmat_
    n_components = model.n_components
    
    for visit_id, visit_df in df.groupby('visit_occurrence_id'):
        visit_df = visit_df.sort_values('t_hr').reset_index(drop=True)
        X = visit_df[feature_cols].values
        n_samples = len(X)
        
        if n_samples == 0:
            continue
        
        # Compute log emission probabilities
        log_emit = np.zeros((n_samples, n_components))
        for k in range(n_components):
            try:
                mvn = multivariate_normal(
                    mean=model.means_[k],
                    cov=model.covars_[k],
                    allow_singular=True
                )
                log_emit[:, k] = mvn.logpdf(X)
            except:
                log_emit[:, k] = -100
        
        # Forward pass for filtered posteriors
        log_startprob = np.log(model.startprob_ + 1e-10)
        log_transmat = np.log(transmat + 1e-10)
        log_alpha = np.zeros((n_samples, n_components))
        
        # Initialize
        log_alpha[0] = log_startprob + log_emit[0]
        
        # Forward recursion
        for t in range(1, n_samples):
            for j in range(n_components):
                log_alpha[t, j] = log_emit[t, j] + np.logaddexp.reduce(
                    log_alpha[t-1] + log_transmat[:, j]
                )
        
        # Normalize to get filtered probabilities
        log_normalizer = np.logaddexp.reduce(log_alpha, axis=1, keepdims=True)
        filtered_probs = np.exp(log_alpha - log_normalizer)
        
        # Inferred phase
        phase_hat = np.argmax(filtered_probs, axis=1)
        
        # 1-hour ahead forecast
        forecast_probs = filtered_probs @ transmat
        phase_forecast = np.argmax(forecast_probs, axis=1)
        
        # Build results
        for idx in range(n_samples):
            result = {
                'visit_occurrence_id': visit_id,
                't_hr': visit_df.loc[idx, 't_hr'],
                'phase_hat': phase_hat[idx],
                'phase_name': PHASE_NAMES[phase_hat[idx]],
            }
            
            # Current probabilities
            for k in range(n_components):
                result[f'prob_{PHASE_NAMES[k]}'] = filtered_probs[idx, k]
            
            # Forecast
            result['phase_hat_tplus1'] = phase_forecast[idx]
            result['phase_name_tplus1'] = PHASE_NAMES[phase_forecast[idx]]
            
            for k in range(n_components):
                result[f'prob_{PHASE_NAMES[k]}_tplus1'] = forecast_probs[idx, k]
            
            results.append(result)
    
    results_df = pd.DataFrame(results)
    print(f"  Generated predictions for {results_df['visit_occurrence_id'].nunique():,} visits")
    print(f"  Total predictions: {len(results_df):,} rows")
    
    return results_df

In [ ]:
# Run inference on all data
predictions_df = infer_and_forecast(model, df, feature_cols)

# Preview
print("\nPredictions preview:")
display(predictions_df.head(10))

In [ ]:
# Save predictions
predictions_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved predictions to: {OUTPUT_CSV}")

## 7. Sanity Checks & Visualization

In [ ]:
# Merge predictions with original features for plotting
df_plot = df_raw.merge(
    predictions_df[['visit_occurrence_id', 't_hr', 'phase_hat', 'phase_name']],
    on=['visit_occurrence_id', 't_hr'],
    how='inner'
)
print(f"Merged {len(df_plot):,} rows for plotting")

In [ ]:
# Plot (a): Average PEEP and Peak Pressure by Phase
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PEEP by phase
if 'peep_mean' in df_plot.columns:
    ax = axes[0]
    phase_peep = df_plot.groupby('phase_name')['peep_mean'].agg(['mean', 'std'])
    phase_peep = phase_peep.reindex(['Acute', 'Recovery', 'Weaning', 'Liberation'])
    bars = ax.bar(phase_peep.index, phase_peep['mean'], 
                  yerr=phase_peep['std'], capsize=5,
                  color=[PHASE_COLORS[p] for p in phase_peep.index])
    ax.set_ylabel('PEEP (cmH2O)')
    ax.set_title('Average PEEP by Inferred Phase')
    ax.set_ylim(bottom=0)

# Peak Pressure by phase
if 'peak_mean' in df_plot.columns:
    ax = axes[1]
    phase_peak = df_plot.groupby('phase_name')['peak_mean'].agg(['mean', 'std'])
    phase_peak = phase_peak.reindex(['Acute', 'Recovery', 'Weaning', 'Liberation'])
    bars = ax.bar(phase_peak.index, phase_peak['mean'],
                  yerr=phase_peak['std'], capsize=5,
                  color=[PHASE_COLORS[p] for p in phase_peak.index])
    ax.set_ylabel('Peak Pressure (cmH2O)')
    ax.set_title('Average Peak Pressure by Inferred Phase')
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('hmm_vitals_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Sanity check: Acute phase should have highest PEEP/peak, Weaning/Liberation lowest")

In [ ]:
# Plot (b): Sample Visit Timelines
# Select visits with good length and phase variation
visit_lengths = predictions_df.groupby('visit_occurrence_id').size()
visit_phase_variety = predictions_df.groupby('visit_occurrence_id')['phase_hat'].nunique()

good_visits = visit_lengths[(visit_lengths >= 12) & (visit_phase_variety >= 2)].index.tolist()
if len(good_visits) == 0:
    good_visits = visit_lengths[visit_lengths >= 6].index.tolist()

n_samples = min(3, len(good_visits))
np.random.seed(42)
sample_visits = np.random.choice(good_visits, size=n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 1, figsize=(14, 3*n_samples))
if n_samples == 1:
    axes = [axes]

for idx, visit_id in enumerate(sample_visits):
    ax = axes[idx]
    visit_data = predictions_df[predictions_df['visit_occurrence_id'] == visit_id].sort_values('t_hr')
    
    # Plot phase over time as colored scatter/line
    for phase in PHASE_NAMES.values():
        phase_data = visit_data[visit_data['phase_name'] == phase]
        ax.scatter(phase_data['t_hr'], phase_data['phase_hat'], 
                   c=PHASE_COLORS[phase], s=80, label=phase, zorder=3)
    
    # Connect with line
    ax.plot(visit_data['t_hr'], visit_data['phase_hat'], 'k-', alpha=0.3, zorder=1)
    
    ax.set_yticks([0, 1, 2, 3])
    ax.set_yticklabels(['Acute', 'Recovery', 'Weaning', 'Liberation'])
    ax.set_xlabel('Hours since intubation')
    ax.set_ylabel('Phase')
    ax.set_title(f'Visit {visit_id} - Phase Trajectory ({len(visit_data)} hours)')
    ax.grid(True, alpha=0.3)
    
    if idx == 0:
        ax.legend(loc='upper right', ncol=4)

plt.tight_layout()
plt.savefig('hmm_sample_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Phase distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
ax = axes[0]
phase_counts = predictions_df['phase_name'].value_counts()
phase_counts = phase_counts.reindex(['Acute', 'Recovery', 'Weaning', 'Liberation'])
bars = ax.bar(phase_counts.index, phase_counts.values, 
              color=[PHASE_COLORS[p] for p in phase_counts.index])
ax.set_ylabel('Count')
ax.set_title('Distribution of Inferred Phases')
total = phase_counts.sum()
for bar, count in zip(bars, phase_counts.values):
    ax.annotate(f'{count/total*100:.1f}%', 
                xy=(bar.get_x() + bar.get_width()/2, count),
                ha='center', va='bottom')

# Phase transitions (actual vs forecast)
ax = axes[1]
# Calculate forecast accuracy
forecast_matches = []
for visit_id in predictions_df['visit_occurrence_id'].unique():
    visit_preds = predictions_df[predictions_df['visit_occurrence_id'] == visit_id].sort_values('t_hr')
    if len(visit_preds) < 2:
        continue
    for i in range(len(visit_preds) - 1):
        forecast = visit_preds.iloc[i]['phase_hat_tplus1']
        actual_next = visit_preds.iloc[i + 1]['phase_hat']
        forecast_matches.append(forecast == actual_next)

accuracy = np.mean(forecast_matches) if forecast_matches else 0
ax.bar(['Correct', 'Incorrect'], [accuracy, 1-accuracy], 
       color=['#2ca02c', '#d62728'])
ax.set_ylabel('Proportion')
ax.set_title(f'1-Hour Forecast Accuracy: {accuracy:.1%}')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('hmm_phase_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary & Results

In [ ]:
print("=" * 70)
print("HMM VENTILATION PHASE INFERENCE - SUMMARY")
print("=" * 70)

print(f"\n📊 Data:")
print(f"   Total visits: {predictions_df['visit_occurrence_id'].nunique():,}")
print(f"   Total hourly predictions: {len(predictions_df):,}")
print(f"   Train/Test split: {len(train_visits):,} / {len(test_visits):,} visits")

print(f"\n🔧 Model:")
print(f"   HMM components: {N_COMPONENTS}")
print(f"   Features used: {len(feature_cols)}")
print(f"   Covariance type: {COVARIANCE_TYPE}")

print(f"\n📈 Phase Distribution:")
for phase, count in predictions_df['phase_name'].value_counts().items():
    pct = count / len(predictions_df) * 100
    print(f"   {phase}: {count:,} ({pct:.1f}%)")

print(f"\n🎯 1-Hour Forecast Accuracy: {accuracy:.1%}")

print(f"\n💾 Outputs:")
print(f"   Predictions saved to: {OUTPUT_CSV}")
print(f"   Plots saved: hmm_vitals_by_phase.png, hmm_sample_trajectories.png, hmm_phase_distribution.png")

print("\n" + "=" * 70)
print("Pipeline complete! ✅")

---
## How to Run

1. **Install dependencies:**
   ```bash
   pip install hmmlearn scikit-learn pandas numpy matplotlib scipy
   ```

2. **Update configuration:**
   - Set `INPUT_CSV` to your data file path
   - Adjust threshold cutoffs if needed (PEEP_ACUTE_THRESHOLD, etc.)

3. **Run all cells** or use the script version:
   ```bash
   python hmm_ventilation_phase_inference.py
   ```

4. **Outputs:**
   - `hmm_phase_predictions.csv` - Per-row predictions with phase probabilities
   - Sanity check plots (PNG files)

---

## Limitations & Caveats

1. **Latent phases are proxies:** Without direct ventilator mode, FiO2, or SBT results, the HMM learns "latent course stages" correlated with available measurements.

2. **Filtered vs smoothed inference:** We implement a forward-pass for true filtered posteriors P(S_t | x_0:t). Standard `predict_proba` gives smoothed P(S_t | x_0:T).

3. **Weak label initialization:** The heuristic labels seed the HMM but the model refines these through EM learning.

4. **Transition matrix:** Initialized with clinical priors (forward progression), can be fixed or learned from data.